In [41]:
# testing hysteresis parameter calculation? 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 

from scipy.optimize import curve_fit
from scipy.integrate import quad
from scipy.stats import linregress

storm_directory = 'data/constituents'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('data/shear_stress/average_shear_stress_lajara_full.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('data/sonde_data/sonde_down_full_record.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('data/sonde_data/sonde_up_full_record.csv', parse_dates=['DateTime'], index_col='DateTime')


Process and Merge Data

In [42]:
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

In [43]:
# join shear stress data and the matching sonde data with the storm data
merged_storms = {}

for storm_name, storm_df in storms.items():
    if "down" in storm_name.lower():
        sonde_df = sonde_downstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    elif "up" in storm_name.lower():
        sonde_df = sonde_upstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    else:
        continue

    merged_storm_df = storm_df.join(shear_stress, how="left")
    merged_storm_df = merged_storm_df.join(sonde_df, how="left")
    merged_storms[storm_name] = merged_storm_df

# also join the sonde data with the shear stress data for the entire record 
sonde_downstream = shear_stress.join(sonde_downstream[["Turbidity FNU", "fDOM RFU"]], how="left")
sonde_upstream = shear_stress.join(sonde_upstream[["Turbidity FNU", "fDOM RFU"]], how="left")

In [44]:
merged_storms['st1_up']

,SS (uL/L),SSC (mg/L),SRP (mg/L),TP (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L),shear_stress,Turbidity FNU,fDOM RFU
Date_Time,,,,,,,,,,
2021-07-23 14:30:00,0.000,4.409565,0.02625,0.03125,0.00,2.864641,1.754,68.223515,3.000,15.840
2021-07-23 14:54:00,92.060,68.981000,0.04200,0.09400,0.15,11.105057,4.522,83.757128,8.208,16.488
2021-07-23 15:02:00,95.690,124.475000,0.04500,0.12000,0.28,29.712121,4.436,87.915423,12.808,16.984
2021-07-23 16:13:00,403.515,359.220000,0.09000,0.36200,1.66,41.279351,12.442,126.540663,122.774,19.432
2021-07-23 16:58:00,124.040,86.111000,0.08300,0.20800,0.38,9.524065,12.977,129.045967,42.612,31.564
2021-07-23 17:58:00,65.300,37.228000,0.06700,0.13200,0.13,6.538644,13.108,110.251417,21.154,37.150
2021-07-23 20:13:00,42.870,30.435000,0.04800,0.08600,0.06,5.707480,12.073,90.089846,10.370,39.008


Hysteresis index calculations functions

In [45]:
## regression equations
# rising limb: logarithmic
def log_func(Q, a, b):
    return a * np.log(Q) + b
# falling limb: exponential
def exp_func(Q, a, b):
    return a * np.exp(b * Q)

# split hydrograph into rising and falling limbs based on peak flow
def split_hydrograph(df, q_col):
    # if df empty or q_col has no valid values, return empty limbs
    if df.empty or df[q_col].dropna().empty:
        return df.copy(), df.copy()
    peak_time = df[q_col].idxmax()
    rising = df.loc[:peak_time].copy()
    falling = df.loc[peak_time:].copy()
    return rising, falling

# fit curves and calculate R²
def fit_curve(x, y, func, p0=None):
    mask = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x[mask])
    y = np.asarray(y[mask])
    if len(x) < 2:  # need at least 2 points
        return None, None
    try:
        popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000)
    except Exception:
        return None, None
    y_pred = func(x, *popt)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    if np.isclose(ss_tot, 0):
        r2 = np.nan
    else:
        r2 = 1 - (ss_res / ss_tot)
    return popt, r2

# Langlois 2025 H calculation
def compute_langlois_H(event_df, tau_col, constituent_col, r2_threshold=0.50, storm_name=None):

    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty or df[tau_col].dropna().empty or df[constituent_col].dropna().empty:
        print(f"No valid {tau_col} or {constituent_col} for {storm_name or 'unknown'}; skipping")
        return None

    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        print(f"No rising or falling limb for {storm_name or 'unknown'}; skipping")
        return None

    # discharge overlap range
    tau_min = max(rising[tau_col].min(), falling[tau_col].min())
    tau_max = min(rising[tau_col].max(), falling[tau_col].max())

    # fit rising limb
    try:
        rise_params, rise_r2 = fit_curve( rising[tau_col].values, rising[constituent_col].values,
            log_func, p0=[1, 1])
    except:
        return None

    if rise_params is None:
        return None

    # fit falling limb
    try:
        fall_params, fall_r2 = fit_curve(falling[tau_col].values, falling[constituent_col].values,
            exp_func, p0=[1, -0.01])
    except:
        return None

    if fall_params is None:
        return None

    # check fit quality with r2 threshold
    if (rise_r2 < r2_threshold) or (fall_r2 < r2_threshold):
            label = storm_name if storm_name is not None else "unknown storm"
            print(f"Poor fit for rising (R²={rise_r2:.2f}) or falling (R²={fall_r2:.2f}) limb in {label} for {constituent_col}")

    # integrated areas
    rise_area, _ = quad(lambda Q: log_func(Q, *rise_params), tau_min, tau_max)
    fall_area, _ = quad(lambda Q: exp_func(Q, *fall_params), tau_min, tau_max)

    # hysteresis index
    H = rise_area / fall_area
    return {
        'H': H,
        'rise_r2': rise_r2,
        'fall_r2': fall_r2,
        'rise_area': rise_area,
        'fall_area': fall_area,
        'rise_params': rise_params,
        'fall_params': fall_params,
        'tau_min': tau_min,
        'tau_max': tau_max
    }

Calculate H for all events

In [46]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        result = compute_langlois_H(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/langlois_hysteresis_results.csv', index=False)

Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.35) or falling (R²=0.96) limb in st1_down for POC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.26) or falling (R²=0.94) limb in st1_up for POC (mg/L)
Poor fit for rising (R²=0.35) or falling (R²=0.99) limb in st2_down for SSC (mg/L)
Poor fit for rising (R²=0.34) or falling (R²=0.99) limb in st2_down for PP (mg/L)
Poor fit for rising (R²=0.27) or falling (R²=0.89) limb in st2_down for POC (mg/L)
Poor fit for rising (R²=0.85) or falling (R²=0.47) limb in st2_down for DOC (mg/L)
Poor fit for rising (R²=0.75) or falling (R²=0.19) limb in st3_down for PP (mg/L)
Poor fit for rising (R²=0.82) or falling (R²=0.06) limb in st3_down for DOC (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.06) or falling (R²=0.71) limb in st4_down for POC (mg/L)
Poor fit f

C:\Users\nicol\AppData\Local\Temp\ipykernel_3100\4005652132.py:27: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000)


### Plots

In [53]:
def plot_langlois_hysteresis(event_df, tau_col,
    constituent_col, r2_threshold=0.7, storm_name=None, out_dir='HI_calculations/langlois_plots'):

    df = event_df[[tau_col, constituent_col]].dropna()
    rising, falling = split_hydrograph(df, tau_col)

    # fit curves and calculate R²
    rise_params, rise_r2 = fit_curve(rising[tau_col].values,
        rising[constituent_col].values, log_func, p0=[1, 1])

    if rise_params is None:
        return

    fall_params, fall_r2 = fit_curve(falling[tau_col].values,
        falling[constituent_col].values, exp_func, p0=[1, -0.01])

    if fall_params is None:
        return

    tau_min = max(rising[tau_col].min(), falling[tau_col].min())
    tau_max = min(rising[tau_col].max(), falling[tau_col].max())
    tau_fit = np.linspace(tau_min, tau_max, 200)

    rise_fit = log_func(tau_fit, *rise_params)
    fall_fit = exp_func(tau_fit, *fall_params)

    # calculate area under the curve
    rise_area, _ = quad(lambda Q: log_func(Q, *rise_params),
        tau_min, tau_max)

    fall_area, _ = quad(lambda Q: exp_func(Q, *fall_params),
        tau_min, tau_max)
    
    H = rise_area / fall_area

    # PLOT 
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # time series
    ax = axes[0]
    ax.plot(df.index, df[tau_col], label='Shear stress', color='tab:blue')
    ax.set_ylabel(tau_col, color='tab:blue')
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    ax2 = ax.twinx()
    ax2.plot(
        df.index, df[constituent_col], label=constituent_col, color='tab:red')
    ax2.set_ylabel(constituent_col, color='tab:red')
    ax.set_title('Event Time Series')

    # hysteresis loop
    ax = axes[1]
    ax.scatter(rising[tau_col], rising[constituent_col], label='Rising limb', color='tab:orange')
    ax.scatter(falling[tau_col], falling[constituent_col], label='Falling limb', color='tab:green')
    ax.plot(tau_fit, rise_fit, linewidth=2, label=f'Rising fit (R²={rise_r2:.2f})', color='tab:orange')
    ax.plot(tau_fit, fall_fit, linewidth=2, label=f'Falling fit (R²={fall_r2:.2f})', color='tab:green')

    ax.set_xlabel(tau_col)
    ax.set_ylabel(constituent_col)
    ax.set_title(f'H = {H:.2f}')
    ax.legend()

    # add a main title for the whole figure
    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    plt.suptitle(main_title, fontsize=16)
    plt.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    fname = f"{storm_name}_{constituent_col}_langlois.png".replace(" ", "_").replace("/", "_")
    plt.savefig(os.path.join(out_dir, fname), dpi=300, bbox_inches='tight')
    plt.close(fig)

In [48]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        plot_langlois_hysteresis(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

C:\Users\nicol\AppData\Local\Temp\ipykernel_3100\4005652132.py:27: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000)


## Turbidity and fDOM hysteresis 

Separating storm by date and time

In [49]:
# define storm windows once
storm_windows = {
    "st1_down": ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st1_up":   ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st2_down": ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st2_up":   ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st3_down": ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st3_up":   ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st4_down": ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st4_up":   ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st5_down": ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st5_up":   ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st6_down": ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st6_up":   ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st7_down": ("2023-09-14 13:00", "2023-09-15 13:00"),
    "st7_up":   ("2023-09-14 13:00", "2023-09-15 13:00")
}

# build event dictionary directly (no function)
sonde_events = {}
down = sonde_downstream.sort_index()   
up = sonde_upstream.sort_index()      

for storm_name, (start, end) in storm_windows.items():
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if "down" in storm_name.lower():
        src = down
    elif "up" in storm_name.lower():
        src = up
    else:
        continue
    sonde_events[storm_name] = src.loc[(src.index >= start) & (src.index <= end)].copy()

Calculate HI for sonde data

In [50]:
sonde_results = []

for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        result = compute_langlois_H(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            sonde_results.append(result)

sonde_results = pd.DataFrame(sonde_results)
sonde_results.to_csv('HI_calculations/langlois_sonde_hysteresis_results.csv', index=False)

Poor fit for rising (R²=0.53) or falling (R²=0.43) limb in st1_down for Turbidity FNU
Poor fit for rising (R²=0.47) or falling (R²=0.38) limb in st1_down for fDOM RFU
Poor fit for rising (R²=0.47) or falling (R²=0.43) limb in st1_up for fDOM RFU
Poor fit for rising (R²=0.20) or falling (R²=0.63) limb in st2_down for Turbidity FNU
Poor fit for rising (R²=0.53) or falling (R²=0.03) limb in st2_down for fDOM RFU
No valid shear_stress or Turbidity FNU for st2_up; skipping
No valid shear_stress or fDOM RFU for st2_up; skipping
No valid shear_stress or Turbidity FNU for st3_up; skipping
No valid shear_stress or fDOM RFU for st3_up; skipping
Poor fit for rising (R²=0.04) or falling (R²=0.12) limb in st4_down for fDOM RFU
Poor fit for rising (R²=0.86) or falling (R²=0.46) limb in st4_up for Turbidity FNU
Poor fit for rising (R²=0.34) or falling (R²=0.08) limb in st4_up for fDOM RFU
Poor fit for rising (R²=0.17) or falling (R²=0.98) limb in st5_down for Turbidity FNU
No valid shear_stress or Tu

In [51]:
sonde_results

,H,rise_r2,fall_r2,rise_area,fall_area,rise_params,fall_params,tau_min,tau_max,storm,constituent
0,2.154029,0.528522,0.431160,4709.168041,2186.214329,"[199.11117480709814, -841.8807892687462]","[0.7280974093235647, 0.03587642161002283]",65.947067,133.064651,st1_down,Turbidity FNU
1,0.533001,0.472196,0.376194,1710.324276,3208.858353,"[15.823733431202651, -46.99929717095012]","[9.707141042852687, 0.015569625803846916]",65.947067,133.064651,st1_down,fDOM RFU
2,3.021200,0.658847,0.974220,3843.079373,1272.037280,"[165.7794096331003, -702.9610366832732]","[0.18427500194040966, 0.043145656370022295]",68.376213,133.471429,st1_up,Turbidity FNU
3,0.579534,0.470854,0.430098,1210.837006,2089.330098,"[11.975170746304244, -36.44241034149741]","[8.081813922520697, 0.013354996011438882]",68.376213,133.471429,st1_up,fDOM RFU
4,1.278401,0.197736,0.627715,107.838013,84.353853,"[-62.37996893622246, 278.0394854685231]","[0.01367344575323951, 0.09988221586226206]",64.042231,71.117322,st2_down,Turbidity FNU
5,0.761437,0.534219,0.032525,203.108779,266.743862,"[121.4278304844523, -482.8498558795409]","[29.593143268548282, 0.003583012285960271]",64.042231,71.117322,st2_down,fDOM RFU
6,1.303260,0.911872,0.921327,124.262291,95.347275,"[88.34829384049068, -362.40546728880247]","[0.00015294907139677384, 0.15698453538585908]",64.885214,74.728309,st3_down,Turbidity FNU
7,0.649963,0.886737,0.568429,235.871315,362.899462,"[69.6937173569455, -271.8798184932581]","[82.25903032591263, -0.011503924069268753]",64.885214,74.728309,st3_down,fDOM RFU
8,1.536835,0.859524,0.714889,365.909758,238.093030,"[172.11696764897766, -711.4143643566383]","[0.0011057865120498568, 0.1306720028197271]",65.909725,79.767296,st4_down,Turbidity FNU
9,0.712207,0.037487,0.117765,107.789173,151.345335,"[2.300216584688032, -2.08205315917614]","[1.5865890269844334, 0.026408722355405704]",65.909725,79.767296,st4_down,fDOM RFU


Plot

In [56]:
all_results = []

for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        plot_langlois_hysteresis(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name,
            out_dir='HI_calculations/langlois_plots/sonde')